In [ ]:

# INSTALL

!pip install laspy lazrs rasterio scikit-learn torch cloth-simulation-filter -q

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 649.3/649.3 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.7 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:

# IMPORTS

import os
import gc
import laspy
import CSF as M
from CSF import CSF
import numpy as np
from sklearn.neighbors import KDTree
from scipy.interpolate import RBFInterpolator
from scipy.ndimage import uniform_filter
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.transform import rowcol
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error


# PATHS

ROOT_DIR   = "/content/drive/MyDrive/SVAMITVA_Data"
OUTPUT_DIR = "/content/drive/MyDrive/SVAMITVA_Data_DTM_MULTI"

LAZ_FILES = [
    "/content/drive/MyDrive/SVAMITVA_Data/Rajasthan_Point_Cloud/Rajasthan_Point_Cloud/67169_5NKR_CHAKHIRASINGH.las",
    "/content/drive/MyDrive/SVAMITVA_Data/Punjab_Point_Cloud/Punjab_Point_Cloud/Dhal_Hoshiarpur_31235.las",
    "/content/drive/MyDrive/SVAMITVA_Data/Punjab_Point_Cloud/Gujrat_Point_Cloud/KHAPRETA_510206.laz",
    "/content/drive/MyDrive/SVAMITVA_Data/Punjab_Point_Cloud/Punjab_Point_Cloud/DHUNDA_FATEHGARH SAHIB_32619.laz",
    "/content/drive/MyDrive/SVAMITVA_Data/Rajasthan_Point_Cloud/Rajasthan_Point_Cloud/64334_2H (REFLIGHT)_POINT CLOUD.LAS",
    "/content/drive/MyDrive/SVAMITVA_Data/Gujrat_Point_Cloud/Gujrat_Point_Cloud/DEVDI_POINT CLOUD (511671).las",
]
DTM_FILES = [
    "/content/drive/MyDrive/SVAMITVA_Data_DTM_MULTI/67169_5NKR_CHAKHIRASINGH_DTM.tif",
    "/content/drive/MyDrive/SVAMITVA_Data_DTM_MULTI/Dhal_Hoshiarpur_31235_DTM.tif",
    "/content/drive/MyDrive/SVAMITVA_Data_DTM_MULTI/KHAPRETA_510206_DTM.tif",
    "/content/drive/MyDrive/SVAMITVA_Data_DTM_MULTI/DHUNDA_FATEHGARH SAHIB_32619_DTM.tif",
    "/content/drive/MyDrive/SVAMITVA_Data_DTM_MULTI/64334_2H (REFLIGHT)_POINT CLOUD_DTM.tif",
    "/content/drive/MyDrive/SVAMITVA_Data_DTM_MULTI/DEVDI_POINT CLOUD (511671)_DTM.tif",
]

# KHAPRETA is index 2 — held out for testing, never trained on
TEST_IDX   = 2


# PARAMETERS

MAX_POINTS   = 400_000   # raised from 300k — multi-village needs more coverage
BLOCK_SIZE   = 1024
BATCH_SIZE   = 4
BLOCK_M      = 10.0
OVERLAP      = 0.5
K_NEIGHBORS  = 32
ACCUM_STEPS  = 8
IS_GROUND_COL = None     # set after extract_features

In [ ]:

# ROUGHNESS WEIGHT COMPUTATION

def compute_roughness_map(dtm_path):
    """
    Compute per-cell roughness for a DTM.
    Returns roughness_map (same shape as DTM) and rasterio transform.
    Used to down-weight noisy training cells in the loss function.
    """
    with rasterio.open(dtm_path) as src:
        dtm       = src.read(1).astype(np.float32)
        nodata    = src.nodata
        transform = src.transform
        if nodata is not None:
            dtm[dtm == nodata] = np.nan

    valid_mask              = ~np.isnan(dtm)
    dtm_filled              = dtm.copy()
    dtm_filled[~valid_mask] = np.nanmean(dtm) if valid_mask.any() else 0.0
    dtm_smooth              = uniform_filter(dtm_filled, size=5)
    roughness_map           = np.abs(dtm_filled - dtm_smooth)

    # NaN regions get max roughness so they are ignored in training
    roughness_map[~valid_mask] = 9999.0

    return roughness_map, transform


def sample_roughness_weights(xyz, roughness_map, transform,
                             low_thresh=0.05, high_thresh=0.15):
    """
    Sample roughness value at each point location and convert to loss weight.

    Weight schedule:
      roughness < low_thresh  → weight = 1.0  (fully reliable)
      low_thresh to high_thresh → weight = 0.5 (moderate)
      roughness > high_thresh → weight = 0.1  (noisy, almost ignored)
    """
    rows_idx, cols_idx = rowcol(transform, xyz[:, 0], xyz[:, 1])
    rows_idx = np.clip(rows_idx, 0, roughness_map.shape[0] - 1)
    cols_idx = np.clip(cols_idx, 0, roughness_map.shape[1] - 1)

    roughness_vals = roughness_map[rows_idx, cols_idx]

    weights = np.ones(len(xyz), dtype=np.float32)
    weights[(roughness_vals >= low_thresh) &
            (roughness_vals < high_thresh)] = 0.5
    weights[roughness_vals >= high_thresh]  = 0.1
    weights[roughness_vals >= 999.0]        = 0.0   # NaN regions

    return weights.astype(np.float32)

In [ ]:

# LOAD POINTS — ALL CLASSES + CSF

def load_points(laz_file, max_points=MAX_POINTS):
    las            = laspy.read(laz_file)
    xyz            = np.vstack((las.x, las.y, las.z)).T.astype(np.float32)
    print(f"  Raw points: {len(xyz):,}")

    # Stratified spatial sampling
    idx = None
    if len(xyz) > max_points:
        x_bins   = np.digitize(xyz[:,0],
                       np.linspace(xyz[:,0].min(), xyz[:,0].max(), 50))
        y_bins   = np.digitize(xyz[:,1],
                       np.linspace(xyz[:,1].min(), xyz[:,1].max(), 50))
        cell_ids = x_bins * 100 + y_bins
        unique_cells = np.unique(cell_ids)
        chosen   = []
        per_cell = max_points // len(unique_cells)
        for cell in unique_cells:
            cell_idx = np.where(cell_ids == cell)[0]
            n = min(len(cell_idx), max(1, per_cell))
            chosen.append(np.random.choice(cell_idx, n, replace=False))
        idx = np.concatenate(chosen)
        if len(idx) > max_points:
            idx = np.random.choice(idx, max_points, replace=False)
        xyz = xyz[idx]

    del x_bins, y_bins, cell_ids
    gc.collect()

    print(f"  After sampling: {len(xyz):,}")
    print(f"  Running CSF...")

    # CSF ground detection
    csf_filter = CSF()
    csf_filter.params.bSloopSmooth     = True
    csf_filter.params.cloth_resolution = 0.5
    csf_filter.params.rigidness        = 2
    csf_filter.params.time_step        = 0.65
    csf_filter.params.class_threshold  = 0.4
    csf_filter.params.interations      = 500

    csf_filter.setPointCloud(xyz.tolist())
    ground_idx    = M.VecInt()
    nonground_idx = M.VecInt()
    csf_filter.do_filtering(ground_idx, nonground_idx, False)

    is_ground = np.zeros(len(xyz), dtype=np.float32)
    is_ground[np.array(ground_idx)] = 1.0
    print(f"  CSF ground: {int(is_ground.sum()):,} / {len(xyz):,} "
          f"({100*is_ground.mean():.1f}%)")

    del ground_idx, nonground_idx
    gc.collect()

    # Extra features
    extras = [is_ground[:, None]]

    if hasattr(las, 'intensity'):
        intensity = np.array(las.intensity, dtype=np.float32)
        if idx is not None: intensity = intensity[idx]
        extras.append(intensity[:, None] / 65535.0)
        del intensity

    if hasattr(las, 'red'):
        r = np.array(las.red,   dtype=np.float32)
        g = np.array(las.green, dtype=np.float32)
        b = np.array(las.blue,  dtype=np.float32)
        if idx is not None: r, g, b = r[idx], g[idx], b[idx]
        r /= 65535.0; g /= 65535.0; b /= 65535.0
        ndvi = (r - g) / (r + g + 1e-6)
        extras.extend([r[:,None], g[:,None], b[:,None], ndvi[:,None]])
        del r, g, b, ndvi

    if hasattr(las, 'return_number') and hasattr(las, 'number_of_returns'):
        ret_num  = np.array(las.return_number,     dtype=np.float32)
        num_rets = np.array(las.number_of_returns, dtype=np.float32).clip(1)
        if idx is not None: ret_num, num_rets = ret_num[idx], num_rets[idx]
        ret_norm = (ret_num - 1) / (num_rets - 1 + 1e-6)
        is_last  = (ret_num == num_rets).astype(np.float32)
        extras.extend([ret_norm[:,None], is_last[:,None]])
        del ret_num, num_rets, ret_norm, is_last

    del las, is_ground
    gc.collect()

    pts = np.hstack([xyz, np.hstack(extras)])
    del xyz, extras
    gc.collect()

    print(f"  Final: {len(pts):,} points | {pts.shape[1]} channels")
    return pts

In [ ]:

# SAMPLE DTM

def sample_dtm(points, dtm_path):
    with rasterio.open(dtm_path) as src:
        band      = src.read(1).astype(np.float32)
        nodata    = src.nodata
        transform = src.transform
        rows, cols = rowcol(transform, points[:,0], points[:,1])
        rows = np.clip(rows, 0, band.shape[0] - 1)
        cols = np.clip(cols, 0, band.shape[1] - 1)
        z_gt = band[rows, cols]
        if nodata is not None:
            z_gt[z_gt == nodata] = np.nan
        return z_gt


# FEATURE EXTRACTION

def extract_features(points, dtm_path):
    global IS_GROUND_COL

    xyz   = points[:, :3]
    extra = points[:, 3:] if points.shape[1] > 3 else None

    tree   = KDTree(xyz[:, :2])
    _, idx = tree.query(xyz[:, :2], k=20)

    neighbors_z       = xyz[idx][:, :, 2]
    mean_z            = neighbors_z.mean(axis=1)
    std_z             = neighbors_z.std(axis=1)
    min_z             = neighbors_z.min(axis=1)
    height_above_mean = xyz[:, 2] - mean_z
    height_above_min  = xyz[:, 2] - min_z

    del neighbors_z, mean_z, min_z, idx
    gc.collect()

    min_xyz      = xyz.min(axis=0)
    max_xyz      = xyz.max(axis=0)
    norm_xyz     = (xyz - min_xyz) / (max_xyz - min_xyz + 1e-6)
    centered_xyz = xyz - xyz.mean(axis=0)

    feat_list = [
        xyz,
        centered_xyz,
        norm_xyz,
        height_above_mean[:, None],
        height_above_min[:, None],
        std_z[:, None],
    ]
    if extra is not None:
        feat_list.append(extra)

    features = np.concatenate(feat_list, axis=1).astype(np.float32)

    del centered_xyz, norm_xyz, height_above_mean, height_above_min, std_z
    gc.collect()

    IS_GROUND_COL = 12
    ground_z = sample_dtm(xyz, dtm_path)
    valid    = ~np.isnan(ground_z)

    print(f"  Valid points: {valid.sum():,} / {len(features):,} "
          f"| Feature dim: {features.shape[1]}")

    return features[valid], ground_z[valid].astype(np.float32), xyz[valid]


# NORMALIZATION

def normalize_features(features, f_mean=None, f_std=None):
    compute = f_mean is None
    if compute:
        f_mean = features.mean(axis=0)
        f_std  = features.std(axis=0) + 1e-6

    normalized = (features - f_mean) / f_std

    # Protect binary columns
    normalized[:, IS_GROUND_COL] = features[:, IS_GROUND_COL]
    if features.shape[1] > 19:
        normalized[:, 19] = features[:, 19]

    return normalized, f_mean, f_std

In [ ]:

# BLOCK CREATION — WITH ROUGHNESS WEIGHTS

def create_blocks(features, targets, points,
                  roughness_weights=None, is_training=True):
    """
    roughness_weights: per-point weight array (0.1 / 0.5 / 1.0)
                       based on DTM reliability at that location.
                       If None, all weights = 1.0
    """
    x_coords = points[:, 0]
    y_coords = points[:, 1]
    x_min, x_max = x_coords.min(), x_coords.max()
    y_min, y_max = y_coords.min(), y_coords.max()
    stride = BLOCK_M * (1.0 - OVERLAP)

    if roughness_weights is None:
        roughness_weights = np.ones(len(points), dtype=np.float32)

    blocks      = []
    block_stats = []

    cx = x_min + BLOCK_M / 2
    while cx <= x_max + BLOCK_M / 2:
        cy = y_min + BLOCK_M / 2
        while cy <= y_max + BLOCK_M / 2:
            in_block = (
                (x_coords >= cx - BLOCK_M / 2) &
                (x_coords <  cx + BLOCK_M / 2) &
                (y_coords >= cy - BLOCK_M / 2) &
                (y_coords <  cy + BLOCK_M / 2)
            )
            mask = np.where(in_block)[0]

            if len(mask) < 64:
                cy += stride
                continue

            block_targets  = targets[mask]
            t_mean         = block_targets.mean()
            t_std          = block_targets.std() + 1e-6
            norm_targets   = (block_targets - t_mean) / t_std
            block_features = features[mask]
            block_rweights = roughness_weights[mask]

            # Roughness-weighted sampling
            # Points with higher DTM reliability sampled more often
            try:
                # Combine terrain roughness + DTM reliability for sampling
                terrain_rough  = np.abs(block_features[:, 11]) + 1e-6
                combined_weight = terrain_rough * block_rweights
                sample_weights  = combined_weight / combined_weight.sum()
                replace         = len(mask) < BLOCK_SIZE
                chosen          = np.random.choice(
                    len(mask), BLOCK_SIZE,
                    replace=replace, p=sample_weights)
            except Exception:
                replace = len(mask) < BLOCK_SIZE
                chosen  = np.random.choice(
                    len(mask), BLOCK_SIZE, replace=replace)

            # Pack: [orig_index | features | roughness_weight | norm_target]
            packed = np.hstack([
                mask[chosen][:, None],          # col 0: original index
                block_features[chosen],          # cols 1..F: features
                block_rweights[chosen][:, None], # col F+1: roughness weight
                norm_targets[chosen][:, None]    # col F+2: target
            ])

            blocks.append(packed)
            block_stats.append((t_mean, t_std))

            cy += stride
        cx += stride

    split_label = "train" if is_training else "test"
    print(f"  Created {len(blocks)} {split_label} blocks "
          f"with {OVERLAP*100:.0f}% overlap")
    return np.array(blocks), block_stats

In [ ]:

# DATASET

class PointDataset(Dataset):
    def __init__(self, blocks, block_stats):
        self.blocks      = blocks
        self.block_stats = block_stats

    def __len__(self):
        return len(self.blocks)

    def __getitem__(self, i):
        pts           = self.blocks[i]
        t_mean, t_std = self.block_stats[i]

        orig_indices  = pts[:, 0].astype(int)
        # features = cols 1 to F (everything except index, rweight, target)
        x             = pts[:, 1:-2].astype(np.float32)
        # roughness weight = second to last column
        rweights      = pts[:, -2].astype(np.float32)
        # target = last column
        y             = pts[:, -1].astype(np.float32)

        tree   = KDTree(x[:, :2])
        _, idx = tree.query(x[:, :2], k=K_NEIGHBORS)
        neighbors = x[idx]

        return (
            torch.tensor(x),
            torch.tensor(neighbors),
            torch.tensor(y),
            torch.tensor(orig_indices),
            torch.tensor([t_mean, t_std], dtype=torch.float32),
            torch.tensor(rweights)
        )

In [ ]:

# MODEL

class TerrainNet(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.p1 = nn.Sequential(nn.Conv1d(in_ch, 32, 1), nn.BatchNorm1d(32),  nn.ReLU())
        self.p2 = nn.Sequential(nn.Conv1d(32, 64, 1),    nn.BatchNorm1d(64),  nn.ReLU())
        self.g1 = nn.Sequential(nn.Conv1d(64, 128, 1),   nn.BatchNorm1d(128), nn.ReLU())
        self.g2 = nn.Sequential(nn.Conv1d(128, 128, 1),  nn.BatchNorm1d(128), nn.ReLU())
        self.n1 = nn.Sequential(nn.Conv2d(in_ch, 32, 1), nn.BatchNorm2d(32),  nn.ReLU())
        self.n2 = nn.Sequential(nn.Conv2d(32, 64, 1),    nn.BatchNorm2d(64),  nn.ReLU())
        self.fc1 = nn.Sequential(nn.Conv1d(512, 256, 1), nn.BatchNorm1d(256), nn.ReLU())
        self.fc2 = nn.Sequential(nn.Conv1d(256, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
                                 nn.Dropout(0.3))
        self.fc3 = nn.Sequential(nn.Conv1d(128, 64, 1),  nn.BatchNorm1d(64),  nn.ReLU())
        self.out  = nn.Conv1d(64, 1, 1)

    def forward(self, x, n):
        B, N, C = x.shape
        px = self.p2(self.p1(x.permute(0, 2, 1)))
        gx = self.g2(self.g1(px))
        g_max  = torch.max(gx,  2, keepdim=True)[0].expand(-1, -1, N)
        g_mean = torch.mean(gx, 2, keepdim=True).expand(-1, -1, N)
        g_std  = torch.std(gx,  2, keepdim=True).expand(-1, -1, N)
        global_feat = torch.cat([g_max, g_mean, g_std], dim=1)
        nx   = torch.max(self.n2(self.n1(n.permute(0, 3, 1, 2))), 3)[0]
        feat = torch.cat([px, global_feat, nx], dim=1)
        return self.out(self.fc3(self.fc2(self.fc1(feat))))


# LOSS — WITH ROUGHNESS WEIGHTING

def huber_gradient_loss(pred, target, features,
                        rweights, delta=0.5, lambda_grad=0.01):
    p = pred.squeeze(1)
    t = target.squeeze(1) if target.dim() == 3 else target

    # Combine CSF ground weight + DTM roughness weight
    # is_ground: ground=2.0, nonground=1.0
    # rweights:  reliable=1.0, moderate=0.5, noisy=0.1
    ground_w   = features[:, :, IS_GROUND_COL] * 1.0 + 1.0
    combined_w = ground_w * rweights
    # Normalize so mean weight = 1.0
    combined_w = combined_w / (combined_w.mean(dim=1, keepdim=True) + 1e-6)

    huber_per_point = F.huber_loss(p, t, delta=delta, reduction='none')
    huber           = (huber_per_point * combined_w).mean()

    if lambda_grad > 0 and p.shape[1] > 1:
        pred_diff = p[:, 1:] - p[:, :-1]
        gt_diff   = t[:, 1:] - t[:, :-1]
        grad_loss = F.mse_loss(pred_diff, gt_diff)
        return huber + lambda_grad * grad_loss

    return huber

In [ ]:

# AUGMENTATION

def augment_batch(x, y):
    B, N, C = x.shape
    device  = x.device

    angle = torch.rand(B, device=device) * 2 * np.pi
    cos_a = torch.cos(angle)
    sin_a = torch.sin(angle)

    x_rot = x.clone()
    # Rotate raw XY — cols 0,1
    x_rot[:, :, 0] = cos_a[:,None] * x[:,:,0] - sin_a[:,None] * x[:,:,1]
    x_rot[:, :, 1] = sin_a[:,None] * x[:,:,0] + cos_a[:,None] * x[:,:,1]
    # Rotate centered XY — cols 3,4
    xc = x[:,:,3].clone(); yc = x[:,:,4].clone()
    x_rot[:,:,3] = cos_a[:,None]*xc - sin_a[:,None]*yc
    x_rot[:,:,4] = sin_a[:,None]*xc + cos_a[:,None]*yc
    # Rotate norm XY — cols 6,7
    xc = x[:,:,6].clone(); yc = x[:,:,7].clone()
    x_rot[:,:,6] = cos_a[:,None]*xc - sin_a[:,None]*yc
    x_rot[:,:,7] = sin_a[:,None]*xc + cos_a[:,None]*yc
    # cols 8+ are scalars — NOT rotated

    z_jitter = (torch.rand(B, N, device=device) - 0.5) * 0.04
    y_aug    = y + z_jitter

    if torch.rand(1) > 0.5:
        drop_mask = torch.rand(B, N, device=device) > 0.10
        for b in range(B):
            kept = drop_mask[b].nonzero(as_tuple=True)[0]
            if len(kept) < N:
                pad     = torch.randint(len(kept), (N - len(kept),))
                all_idx = torch.cat([kept, kept[pad]])
                x_rot[b] = x_rot[b][all_idx]
                y_aug[b] = y_aug[b][all_idx]

    return x_rot, y_aug


# TRAINING

def train(model, train_loader, epochs=30, device='cuda'):
    adam_epochs = int(epochs * 0.7)
    sgd_epochs  = epochs - adam_epochs

    optimizer_adam = torch.optim.AdamW(
        model.parameters(), lr=1e-3,
        betas=(0.95, 0.999), weight_decay=1e-4)
    scheduler_adam = torch.optim.lr_scheduler.OneCycleLR(
        optimizer_adam, max_lr=3e-3,
        steps_per_epoch=max(1, len(train_loader) // ACCUM_STEPS),
        epochs=adam_epochs, pct_start=0.1,
        div_factor=10, final_div_factor=100)

    optimizer_sgd = torch.optim.SGD(
        model.parameters(), lr=5e-5,
        momentum=0.9, nesterov=True, weight_decay=1e-4)
    scheduler_sgd = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_sgd, T_max=sgd_epochs, eta_min=1e-7)

    best_loss  = float('inf')
    best_epoch = 0
    no_improve = 0
    PATIENCE   = 10

    for epoch in range(epochs):
        if epoch < adam_epochs:
            optimizer = optimizer_adam
            scheduler = scheduler_adam
            phase     = 'AdamW'
        else:
            optimizer = optimizer_sgd
            scheduler = scheduler_sgd
            phase     = 'SGD+N'

        model.train()
        total_loss = 0
        optimizer.zero_grad()

        for batch_idx, (x, n, y, _, stats, rw) in enumerate(train_loader):
            x  = x.to(device)
            n  = n.to(device)
            y  = y.to(device)
            rw = rw.to(device)

            x_aug, y_aug = augment_batch(x, y)
            pred = model(x_aug, n)
            loss = huber_gradient_loss(pred, y_aug, x_aug, rw)

            (loss / ACCUM_STEPS).backward()

            if (batch_idx + 1) % ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
                if phase == 'AdamW':
                    scheduler.step()

            total_loss += loss.item()

        # Handle leftover batches
        if len(train_loader) % ACCUM_STEPS != 0:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        if phase == 'SGD+N':
            scheduler.step()

        avg_loss = total_loss / len(train_loader)

        if avg_loss < best_loss - 1e-4:
            best_loss  = avg_loss
            best_epoch = epoch
            no_improve = 0
            torch.save(model.state_dict(), 'best_terrainnet_multi.pth')
        else:
            no_improve += 1

        print(f"Epoch {epoch:3d} [{phase:5s}] | Loss: {avg_loss:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.2e} | "
              f"Best: {best_loss:.4f} (ep {best_epoch})")

        if no_improve >= PATIENCE:
            print(f"  Early stop — no improvement for {PATIENCE} epochs.")
            break

    model.load_state_dict(torch.load('best_terrainnet_multi.pth'))
    print(f"\nTraining complete. Best: {best_loss:.4f} at epoch {best_epoch}")
    return model

In [ ]:

# PREDICTION

def predict(model, blocks, block_stats, device='cuda'):
    model.eval()
    dataset = PointDataset(blocks, block_stats)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    pred_sum   = {}
    pred_count = {}

    with torch.no_grad():
        for x, n, _, indices, stats, rw in loader:
            x   = x.to(device)
            n   = n.to(device)
            out = model(x, n).cpu().numpy()

            for b in range(out.shape[0]):
                t_mean = stats[b, 0].item()
                t_std  = stats[b, 1].item()
                for i in range(out.shape[2]):
                    idx = int(indices[b, i])
                    val = out[b, 0, i] * t_std + t_mean
                    pred_sum[idx]   = pred_sum.get(idx, 0.0) + val
                    pred_count[idx] = pred_count.get(idx, 0)  + 1

    valid_indices = sorted(pred_sum.keys())
    preds = np.array([pred_sum[i] / pred_count[i] for i in valid_indices])
    return preds, valid_indices


# RASTERIZATION

def rasterize_predictions(xyz, valid_indices, preds, targets,
                           resolution=1.0):
    pts = xyz[valid_indices]
    x, y = pts[:, 0], pts[:, 1]
    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()
    cols = int((x_max - x_min) / resolution) + 1
    rows = int((y_max - y_min) / resolution) + 1

    pred_grid = np.full((rows, cols), np.nan, dtype=np.float32)
    gt_grid   = np.full((rows, cols), np.nan, dtype=np.float32)

    col_idx = ((x - x_min) / resolution).astype(int).clip(0, cols-1)
    row_idx = ((y - y_min) / resolution).astype(int).clip(0, rows-1)

    pred_cells = defaultdict(list)
    gt_cells   = defaultdict(list)
    gt_raw     = targets[valid_indices]

    for i in range(len(preds)):
        key = (row_idx[i], col_idx[i])
        pred_cells[key].append(preds[i])
        gt_cells[key].append(gt_raw[i])

    for (r, c), vals in pred_cells.items():
        pred_grid[r, c] = np.median(vals)
    for (r, c), vals in gt_cells.items():
        gt_grid[r, c] = np.median(vals)

    for grid in [pred_grid, gt_grid]:
        known_mask = ~np.isnan(grid)
        if known_mask.sum() < 10:
            continue
        kr, kc       = np.where(known_mask)
        nan_r, nan_c = np.where(~known_mask)
        if len(nan_r) == 0:
            continue
        if len(kr) > 5000:
            sub    = np.random.choice(len(kr), 5000, replace=False)
            kr, kc = kr[sub], kc[sub]
        kv  = grid[kr, kc]
        rbf = RBFInterpolator(
            np.stack([kr, kc], axis=1), kv,
            kernel='thin_plate_spline', smoothing=2.0)
        grid[nan_r, nan_c] = rbf(np.stack([nan_r, nan_c], axis=1))

    return pred_grid, gt_grid

In [ ]:

# MAIN PIPELINE — MULTI-VILLAGE

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Training villages: {len(LAZ_FILES)-1}  |  Test village index: {TEST_IDX}\n")

# ---- Process each village ----
train_blocks      = []
train_block_stats = []
all_f_means       = []
all_f_stds        = []

test_features    = None
test_targets     = None
test_xyz         = None
test_blocks      = None
test_block_stats = None

for i, (laz_path, dtm_path) in enumerate(zip(LAZ_FILES, DTM_FILES)):
    village_name = os.path.basename(laz_path).split('.')[0][:30]
    is_test      = (i == TEST_IDX)
    label        = "TEST" if is_test else "TRAIN"

    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(LAZ_FILES)}] {village_name}  [{label}]")
    print(f"{'='*60}")

    # Load points
    pts = load_points(laz_path)

    # Extract features
    feats, tgts, xyz = extract_features(pts, dtm_path)
    del pts
    gc.collect()

    # Compute roughness weights from DTM
    print(f"  Computing roughness weights...")
    roughness_map, r_transform = compute_roughness_map(dtm_path)
    rweights = sample_roughness_weights(xyz, roughness_map, r_transform)
    print(f"  Roughness weights — reliable: "
          f"{(rweights==1.0).sum():,}  "
          f"moderate: {(rweights==0.5).sum():,}  "
          f"noisy: {(rweights==0.1).sum():,}  "
          f"ignored: {(rweights==0.0).sum():,}")
    del roughness_map
    gc.collect()

    # Normalize features
    if is_test:
        # Test village — normalize using training mean/std
        # We compute its own stats here but will re-normalize after
        # all training villages are processed
        feats_norm, f_mean, f_std = normalize_features(feats)
        test_features    = feats      # keep unnormalized for now
        test_targets     = tgts
        test_xyz         = xyz
        test_f_mean      = f_mean
        test_f_std       = f_std
        test_rweights    = rweights
        print(f"  Test village saved — will normalize with global stats")
        del feats_norm
    else:
        feats_norm, f_mean, f_std = normalize_features(feats)
        all_f_means.append(f_mean)
        all_f_stds.append(f_std)

        # Create training blocks
        blocks, block_stats = create_blocks(
            feats_norm, tgts, xyz,
            roughness_weights=rweights,
            is_training=True)
        train_blocks.extend(blocks.tolist())
        train_block_stats.extend(block_stats)
        print(f"  Running total training blocks: {len(train_blocks)}")

        del feats_norm, feats
    del xyz, tgts, rweights
    gc.collect()

# ---- Compute global normalization stats from training villages ----
print(f"\nComputing global normalization stats from "
      f"{len(all_f_means)} training villages...")
global_f_mean = np.stack(all_f_means).mean(axis=0)
global_f_std  = np.stack(all_f_stds).mean(axis=0)

# Re-normalize test village with global stats
print("Normalizing test village with global stats...")
test_feats_norm, _, _ = normalize_features(
    test_features, f_mean=global_f_mean, f_std=global_f_std)
test_blocks, test_block_stats = create_blocks(
    test_feats_norm, test_targets, test_xyz,
    roughness_weights=test_rweights,
    is_training=False)
del test_features, test_feats_norm, test_rweights
gc.collect()

# ---- Finalize training data ----
train_blocks_arr = np.array(train_blocks)
del train_blocks
gc.collect()

print(f"\nTotal training blocks: {len(train_blocks_arr)}")
print(f"Test blocks:           {len(test_blocks)}")
print(f"Feature dim:           {train_blocks_arr.shape[2] - 3}")
# -3 because packed = [index | features | rweight | target]

# ---- Build dataloader ----
train_dataset = PointDataset(train_blocks_arr, train_block_stats)
train_loader  = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=0, pin_memory=False)
print(f"Train batches per epoch: {len(train_loader)}")

# ---- Build model ----
in_ch = train_blocks_arr.shape[2] - 3  # subtract index, rweight, target
model = TerrainNet(in_ch=in_ch).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# ---- Train ----
print("\nStarting training...")
model = train(model, train_loader, epochs=30, device=device)

# ---- Predict on TEST village ----
print("\nPredicting on test village (KHAPRETA)...")
preds, valid_indices = predict(
    model, test_blocks, test_block_stats, device=device)

# ---- Metrics on test village ----
gt_raw = test_targets[valid_indices]
mae    = mean_absolute_error(gt_raw, preds)
rmse   = np.sqrt(mean_squared_error(gt_raw, preds))
print(f"\nTest village (KHAPRETA) — unseen during training:")
print(f"  MAE:  {mae:.4f} m")
print(f"  RMSE: {rmse:.4f} m")
print(f"  (Previous single-village MAE was 0.117m on same-village eval)")

# ---- Also predict on training villages for comparison ----
# Quick check: re-run on the village we already know
print("\nFor comparison, predicting on training village 67169...")
# This reuses the already-computed blocks for village 0
# (they are inside train_blocks_arr but mixed with others)
# Skip this for brevity — test village result is the important number

# ---- Rasterize test village ----
print("\nRasterizing test village predictions...")
pred_grid, gt_grid = rasterize_predictions(
    test_xyz, valid_indices, preds, test_targets)

# ---- Visualize ----
error_grid = np.abs(pred_grid - gt_grid)
vmax_err   = np.nanpercentile(error_grid, 95)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
im0 = axes[0].imshow(gt_grid,    cmap='terrain', origin='lower')
axes[0].set_title('Ground Truth DTM — KHAPRETA (test village)')
plt.colorbar(im0, ax=axes[0], label='Elevation (m)')

im1 = axes[1].imshow(pred_grid,  cmap='terrain', origin='lower')
axes[1].set_title(f'Predicted DTM  |  MAE={mae:.3f}m  RMSE={rmse:.3f}m')
plt.colorbar(im1, ax=axes[1], label='Elevation (m)')

im2 = axes[2].imshow(error_grid, cmap='hot', origin='lower',
                     vmin=0, vmax=vmax_err)
axes[2].set_title('Absolute Error Map (clipped at p95)')
plt.colorbar(im2, ax=axes[2], label='|Error| (m)')

plt.suptitle(f'Multi-village TerrainNet — Test on unseen village KHAPRETA\n'
             f'Trained on 5 villages  |  MAE={mae:.3f}m  RMSE={rmse:.3f}m',
             fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'multiv_test_results.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ---- Save model ----
torch.save({
    'model_state':  model.state_dict(),
    'f_mean':       global_f_mean,
    'f_std':        global_f_std,
    'in_ch':        in_ch,
    'IS_GROUND_COL': IS_GROUND_COL,
    'train_villages': [os.path.basename(p) for j,p
                       in enumerate(LAZ_FILES) if j != TEST_IDX],
    'test_village':  os.path.basename(LAZ_FILES[TEST_IDX]),
    'test_mae':      float(mae),
    'test_rmse':     float(rmse),
}, os.path.join(OUTPUT_DIR, 'terrainnet_multiv.pth'))
print(f"\nModel saved to {OUTPUT_DIR}/terrainnet_multiv.pth")
print("Done.")

Device: cpu
Training villages: 5  |  Test village index: 2


[1/6] 67169_5NKR_CHAKHIRASINGH  [TRAIN]
  Raw points: 9,839,175
  After sampling: 396,019
  Running CSF...
  CSF ground: 259,417 / 396,019 (65.5%)
  Final: 396,019 points | 11 channels
  Valid points: 268,296 / 396,019 | Feature dim: 20
  Computing roughness weights...
  Roughness weights — reliable: 137,914  moderate: 83,904  noisy: 46,478  ignored: 0
  Created 4518 train blocks with 50% overlap
  Running total training blocks: 4518

[2/6] Dhal_Hoshiarpur_31235  [TRAIN]
  Raw points: 23,431,282
  After sampling: 396,543
  Running CSF...
  CSF ground: 350,783 / 396,543 (88.5%)
  Final: 396,543 points | 11 channels
  Valid points: 345,492 / 396,543 | Feature dim: 20
  Computing roughness weights...
  Roughness weights — reliable: 276,918  moderate: 55,569  noisy: 13,005  ignored: 0
  Created 10029 train blocks with 50% overlap
